In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.ensemble import HistGradientBoostingClassifier

# ─────────────────────────────────────────────
# 1. LOAD DATA & INITIALIZE TIME BUCKETS
# ─────────────────────────────────────────────
def load_and_prepare_data(filepath):
    df = pd.read_csv(filepath, na_values=['NULL', 'Null', 'null', '', 'NaN'])

    timestamp_format = '%Y-%m-%d %H:%M:%S.%f%z'
    for col in ['start_datetime', 'closed_datetime', 'created_date',
                'modified_datetime', 'resolved_datetime']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format=timestamp_format, errors='coerce')

    df = df.dropna(subset=['start_datetime', 'closed_datetime'])

    df['resolution_time_mins'] = (
        df['closed_datetime'] - df['start_datetime']
    ).dt.total_seconds() / 60.0

    df = df[df['resolution_time_mins'] > 0]

    bins   = [0, 30, 90, 240, np.inf]
    labels = [0, 1, 2, 3]          # Quick | Minor | Major | Severe
    df['disruption_category'] = pd.cut(df['resolution_time_mins'], bins=bins, labels=labels)
    df = df.dropna(subset=['disruption_category'])
    return df


# ─────────────────────────────────────────────
# 2. ADVANCED FEATURE ENGINEERING
# ─────────────────────────────────────────────
def engineer_features(df):

    # ── Temporal features ──────────────────────────────────────────────────
    df['start_hour']        = df['start_datetime'].dt.hour
    df['start_day_of_week'] = df['start_datetime'].dt.dayofweek
    df['start_month']       = df['start_datetime'].dt.month
    df['start_quarter']     = df['start_datetime'].dt.quarter
    df['start_week']        = df['start_datetime'].dt.isocalendar().week.astype(int)

    # Cyclical encoding for hour, day-of-week, month
    df['hour_sin']  = np.sin(df['start_hour']        * (2 * np.pi / 24))
    df['hour_cos']  = np.cos(df['start_hour']        * (2 * np.pi / 24))
    df['dow_sin']   = np.sin(df['start_day_of_week'] * (2 * np.pi / 7))
    df['dow_cos']   = np.cos(df['start_day_of_week'] * (2 * np.pi / 7))
    df['month_sin'] = np.sin((df['start_month'] - 1) * (2 * np.pi / 12))
    df['month_cos'] = np.cos((df['start_month'] - 1) * (2 * np.pi / 12))

    # Time-of-day buckets
    df['is_weekend']    = df['start_day_of_week'].isin([5, 6]).astype(int)
    df['is_rush_hour']  = df['start_hour'].apply(lambda x: 1 if (8 <= x <= 11) or (17 <= x <= 20) else 0)
    df['is_night']      = df['start_hour'].apply(lambda x: 1 if x < 6 or x >= 22 else 0)
    df['is_lunch_hour'] = df['start_hour'].apply(lambda x: 1 if 12 <= x <= 14 else 0)

    # Lag from midnight (minutes) — captures how far into the day the event is
    df['mins_from_midnight'] = df['start_hour'] * 60 + df['start_datetime'].dt.minute

    # ── Spatial features ───────────────────────────────────────────────────
    # Distance from Bangalore city centre (approx 12.9716°N, 77.5946°E)
    centre_lat, centre_lon = 12.9716, 77.5946
    df['dist_from_centre'] = np.sqrt(
        (df['latitude']  - centre_lat) ** 2 +
        (df['longitude'] - centre_lon) ** 2
    )

    # Whether start & end coords differ (indicates a moving/spread incident)
    df['has_end_coords'] = (
        (df['endlatitude'].fillna(0) != 0) &
        (df['endlongitude'].fillna(0) != 0)
    ).astype(int)

    df['end_lat_delta'] = (df['endlatitude'].fillna(df['latitude']) - df['latitude']).abs()
    df['end_lon_delta'] = (df['endlongitude'].fillna(df['longitude']) - df['longitude']).abs()
    df['incident_spread'] = np.sqrt(df['end_lat_delta'] ** 2 + df['end_lon_delta'] ** 2)

    # ── Text / description features ────────────────────────────────────────
    for col in ['description', 'comment', 'address', 'end_address']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.lower().replace('nan', '')

    df['desc_length']       = df['description'].str.len()
    df['comment_length']    = df['comment'].str.len()
    df['desc_word_count']   = df['description'].str.split().str.len().fillna(0)
    df['comment_word_count']= df['comment'].str.split().str.len().fillna(0)
    df['has_comment']       = (df['comment_length'] > 0).astype(int)

    # Keyword flags — domain-specific signals
    heavy_kws    = ['heavy', 'truck', 'container', 'tipper', 'bus', 'lorry', 'tanker']
    blocked_kws  = ['blocked', 'jam', 'blocking', 'stopping', 'closed', 'obstruct']
    accident_kws = ['accident', 'crash', 'collision', 'hit', 'skid']
    infra_kws    = ['drain', 'sewer', 'pothole', 'road work', 'dig', 'pipe', 'cement']
    fire_kws     = ['fire', 'smoke', 'burn']
    tree_kws     = ['tree', 'branch', 'fell', 'fallen']

    def kw_flag(series, words):
        return series.apply(lambda x: 1 if any(w in x for w in words) else 0)

    df['flag_heavy']    = kw_flag(df['description'], heavy_kws)
    df['flag_blocked']  = kw_flag(df['description'], blocked_kws)
    df['flag_bmtc']     = df['description'].str.contains('bmtc', na=False).astype(int)
    df['flag_accident'] = kw_flag(df['description'], accident_kws)
    df['flag_infra']    = kw_flag(df['description'], infra_kws)
    df['flag_fire']     = kw_flag(df['description'], fire_kws)
    df['flag_tree']     = kw_flag(df['description'], tree_kws)

    # Combined severity signal
    df['keyword_severity_score'] = (
        df['flag_heavy'] + df['flag_blocked'] + df['flag_accident'] +
        df['flag_fire']  + df['flag_tree']    + df['flag_infra']
    )

    # ── Operational / lifecycle features ──────────────────────────────────
    # Time from report creation to event start (triage lag)
    if 'created_date' in df.columns:
        df['triage_lag_mins'] = (
            df['start_datetime'] - df['created_date']
        ).dt.total_seconds().div(60).clip(lower=0).fillna(0)
    else:
        df['triage_lag_mins'] = 0

    # Whether road closure was required
    df['requires_road_closure_bin'] = (
        df['requires_road_closure'].astype(str).str.lower() == 'true'
    ).astype(int)

    # Historical zone-level average resolution time (target-encoding proxy)
    zone_avg = df.groupby('zone')['resolution_time_mins'].transform('median')
    df['zone_median_resolution'] = zone_avg.fillna(df['resolution_time_mins'].median())

    # Historical junction-level average resolution time
    if 'junction' in df.columns:
        junc_avg = df.groupby('junction')['resolution_time_mins'].transform('median')
        df['junction_median_resolution'] = junc_avg.fillna(df['resolution_time_mins'].median())
    else:
        df['junction_median_resolution'] = df['zone_median_resolution']

    # Event cause encoded ordinally by typical complexity
    cause_severity = {
        'vehicle_breakdown': 1, 'tree_fall': 2, 'others': 2,
        'accident': 3, 'road_work': 3, 'water_logging': 3,
        'unknown': 2
    }
    df['cause_severity'] = (
        df['event_cause'].astype(str).str.lower()
        .map(cause_severity).fillna(2)
    )

    # ── Feature set ────────────────────────────────────────────────────────
    categorical_cols = [
        'event_type', 'event_cause', 'priority', 'zone', 'junction',
        'veh_type', 'corridor', 'direction', 'police_station',
        'reason_breakdown', 'cargo_material', 'status'
    ]
    numerical_cols = [
        # Spatial
        'latitude', 'longitude', 'dist_from_centre',
        'has_end_coords', 'end_lat_delta', 'end_lon_delta', 'incident_spread',
        # Temporal
        'is_weekend', 'is_rush_hour', 'is_night', 'is_lunch_hour',
        'start_day_of_week', 'start_month', 'start_quarter', 'start_week',
        'mins_from_midnight',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
        # Text
        'desc_length', 'comment_length', 'desc_word_count', 'comment_word_count',
        'has_comment',
        # Keyword flags
        'flag_heavy', 'flag_blocked', 'flag_bmtc', 'flag_accident',
        'flag_infra', 'flag_fire', 'flag_tree', 'keyword_severity_score',
        # Operational
        'triage_lag_mins', 'requires_road_closure_bin',
        'zone_median_resolution', 'junction_median_resolution', 'cause_severity',
    ]

    X = df[categorical_cols + numerical_cols].copy()
    y = df['disruption_category'].astype(int)

    for col in categorical_cols:
        X[col] = X[col].astype(str).replace(['nan', 'None', ''], 'Unknown')

    return X, y, categorical_cols, numerical_cols


# ─────────────────────────────────────────────
# 3. LOAD & SPLIT
# ─────────────────────────────────────────────
df_cleaned = load_and_prepare_data('data.csv')
X, y, categorical_cols, numerical_cols = engineer_features(df_cleaned)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ─────────────────────────────────────────────
# 4. PREPROCESSING PIPELINE
# ─────────────────────────────────────────────
# RobustScaler handles outlier-heavy traffic data better than StandardScaler
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# ─────────────────────────────────────────────
# 5. MODEL PIPELINE
# ─────────────────────────────────────────────
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   HistGradientBoostingClassifier(
        random_state=42,
        early_stopping=True,       # prevents over-fitting on small classes
        validation_fraction=0.1,
        n_iter_no_change=20
    ))
])

# ─────────────────────────────────────────────
# 6. EXPANDED HYPERPARAMETER SEARCH
# ─────────────────────────────────────────────
param_grid = {
    'classifier__learning_rate':    [0.03, 0.07, 0.1, 0.15],
    'classifier__max_iter':         [300, 500],
    'classifier__max_depth':        [5, 8, 12],
    'classifier__min_samples_leaf': [10, 20, 40],
    'classifier__l2_regularization':[0.0, 1.0, 5.0],
    'classifier__max_leaf_nodes':   [31, 63],    # NEW: controls tree width
    'classifier__max_bins':         [128, 255],  # NEW: finer splits on continuous features
}

print("Running expanded grid search (this may take a few minutes)...")
tuned_clf = GridSearchCV(
    model_pipeline,
    param_grid=param_grid,
    cv=5,                     # 5-fold for more reliable estimate
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

tuned_clf.fit(X_train, y_train)
print(f"\nBest Parameters: {tuned_clf.best_params_}")
print(f"Best CV Accuracy: {tuned_clf.best_score_ * 100:.2f}%\n")

# ─────────────────────────────────────────────
# 7. EVALUATION
# ─────────────────────────────────────────────
y_pred   = tuned_clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 55)
print("       ENHANCED MODEL PERFORMANCE METRICS")
print("=" * 55)
print(f"  Test Set Accuracy : {accuracy * 100:.2f}%")
print(f"  Best CV Accuracy  : {tuned_clf.best_score_ * 100:.2f}%")
print("=" * 55)

target_names = [
    '<30 mins  (Quick)',
    '30-90 mins (Minor)',
    '90-240 mins (Major)',
    '>240 mins  (Severe)'
]
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

import pickle

# Save the best estimator (recommended)
with open("traffic_disruption_model.pkl", "wb") as f:
    pickle.dump(tuned_clf.best_estimator_, f)

print("Model saved as traffic_disruption_model.pkl")

Running expanded grid search (this may take a few minutes)...
Fitting 5 folds for each of 864 candidates, totalling 4320 fits

Best Parameters: {'classifier__l2_regularization': 5.0, 'classifier__learning_rate': 0.1, 'classifier__max_bins': 255, 'classifier__max_depth': 5, 'classifier__max_iter': 300, 'classifier__max_leaf_nodes': 31, 'classifier__min_samples_leaf': 20}
Best CV Accuracy: 55.40%

       ENHANCED MODEL PERFORMANCE METRICS
  Test Set Accuracy : 56.07%
  Best CV Accuracy  : 55.40%

Classification Report:
                     precision    recall  f1-score   support

  <30 mins  (Quick)       0.47      0.38      0.42       167
 30-90 mins (Minor)       0.51      0.59      0.55       210
90-240 mins (Major)       0.43      0.07      0.12        84
>240 mins  (Severe)       0.68      0.95      0.79       165

           accuracy                           0.56       626
          macro avg       0.52      0.50      0.47       626
       weighted avg       0.53      0.56      0.